# Урок 27. Текст и чат-бот: правила и машинное обучение

**Дата:** 25.03.2027 · **60 минут** · 8–9 класс  
**Модуль 5.** ИИ-приложения (уроки 26–29)  
**Опирается на:** урок 6 (нормализация текста), урок 8 (словарь-счётчик), урок 7 (функции и тесты), уроки 19–20 (классификация, kNN).

Чат-боты отвечают в поддержке банков, магазинов, школ. Самые простые из них работают по **правилам**, более умные — с помощью **машинного обучения**, самые мощные — на **больших языковых моделях**. Сегодня соберём бота для кружка двумя способами — правилами и ML — и честно сравним. А главное — узнаем, как текст превращается в числа, понятные модели.

## Цели урока

К концу занятия ты сможешь:

1. Объяснять, что такое **намерение** (intent) пользователя.
2. Делать бота на правилах и проверять его тестами.
3. Превращать текст в числа методом **«мешок слов»**.
4. Обучать классификатор намерений и добавлять ответ «не понял».
5. Сравнивать правила, ML и большие языковые модели для чат-ботов.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | Как устроен простой чат-бот |
| 5–20 | Бот на правилах и его тесты |
| 20–30 | Мешок слов: текст → числа |
| 30–40 | Бот на машинном обучении, порог уверенности |
| 40–55 | Практика: улучшаем бота |
| 55–60 | Итог и домашнее задание |

## 1. Как устроен простой чат-бот

Пользователь пишет что угодно: «Привет!», «а во сколько завтра кружок?», «где кабинет?». Бот делает два шага:

1. **Понять намерение** (*intent*) — чего хочет человек: поздороваться, узнать расписание, найти кабинет… Это задача **классификации текста**: на входе фраза, на выходе — класс-намерение.
2. **Ответить** заготовленным текстом для этого намерения.

Наш бот будет понимать 5 намерений: приветствие, расписание, место, домашка, прощание. Ответы на них — словарь:

In [ ]:
replies = {
    "приветствие": "Привет! Я бот кружка. Спроси про расписание, кабинет или домашку.",
    "расписание": "Занятия по четвергам в 16:00.",
    "место": "Кружок проходит в кабинете 214, второй этаж.",
    "домашка": "Домашнее задание лежит в конце ноутбука урока.",
    "прощание": "Пока! До встречи в четверг.",
    "не понял": "Не понял вопрос. Спроси про расписание, кабинет или домашку.",
}

## 2. Бот на правилах

Самый простой способ понять намерение — искать **ключевые слова**. Сначала нормализуем текст, как на уроке 6, потом проверяем слова.

In [ ]:
def normalize(text):
    """Нижний регистр и без знаков препинания."""
    text = text.lower()
    for mark in ",.!?;:":
        text = text.replace(mark, " ")
    return text

def rule_intent(text):
    """Определить намерение по ключевым словам."""
    words = normalize(text).split()
    for w in words:
        if w in ["привет", "здравствуйте", "здравствуй", "добрый"]:
            return "приветствие"
        if w in ["когда", "расписание", "сколько"]:
            return "расписание"
        if w in ["где", "кабинет", "кабинете", "адрес"]:
            return "место"
        if w in ["задали", "домашка", "задание"]:
            return "домашка"
        if w in ["пока", "свидания"]:
            return "прощание"
    return "не понял"

for phrase in ["Привет!", "Когда занятие?", "Где кабинет?", "Что задали?", "Пока!"]:
    intent = rule_intent(phrase)
    print(f"{phrase:15} → {intent:12} → {replies[intent]}")

### Тестируем как инженеры

Бот работает на фразах, которые мы сами придумали. А на других? Соберём **тестовые фразы** — так, как их могли бы написать настоящие ученики, — с правильными намерениями.

In [ ]:
test_phrases = [
    "привет бот", "добрый день всем", "хай",
    "во сколько занятие", "в какой день кружок",
    "в каком кабинете урок", "куда идти на занятие",
    "что задали на дом", "есть домашнее задание",
    "до свидания бот", "всем пока", "до встречи",
]
test_intents = [
    "приветствие", "приветствие", "приветствие",
    "расписание", "расписание",
    "место", "место",
    "домашка", "домашка",
    "прощание", "прощание", "прощание",
]

def run_tests(intent_function):
    """Прогнать тесты и вернуть долю верных ответов."""
    correct = 0
    for i, phrase in enumerate(test_phrases):
        got = intent_function(phrase)
        if got == test_intents[i]:
            correct += 1
        else:
            print(f"ОШИБКА: «{phrase}» → {got}, а нужно {test_intents[i]}")
    return correct / len(test_phrases)

print("Доля верных ответов бота на правилах:", round(run_tests(rule_intent), 2))

Правила ломаются на любом слове, которого нет в списке: «хай», «куда», «встречи», «домашнее». Можно дописывать слова бесконечно, но люди всегда придумают новое. Это главная слабость правил (урок 1).

## 3. Мешок слов: как текст превращается в числа

Модели работают с числами. Как превратить фразу в числа? Самый простой способ — **мешок слов** (*bag of words*): посчитать, сколько раз каждое слово встречается во фразе, **не обращая внимания на порядок**, — как будто слова высыпали в мешок.

Это знакомый словарь-счётчик из урока 8:

In [ ]:
def bag_of_words(text):
    counts = {}
    for word in normalize(text).split():
        if word in counts:
            counts[word] += 1
        else:
            counts[word] = 1
    return counts

print(bag_of_words("Когда занятие? Занятие в четверг?"))

Для модели нужно, чтобы у всех фраз были **одинаковые** признаки. Поэтому составляют **словарь** всех слов из обучающих фраз, и каждая фраза превращается в строку чисел: сколько раз в ней встретилось каждое слово словаря. Так текст становится таблицей — как наши таблицы учеников.

В sklearn это делает `CountVectorizer`:

- `fit_transform(тексты)` — составить словарь по обучающим текстам и превратить их в числа;
- `transform(тексты)` — превратить **новые** тексты в числа по **тому же** словарю.

## 4. Обучающие данные

Для обучения нужны **размеченные** примеры: фраза + правильное намерение (урок 17). Их пишут люди — обычно это самая долгая часть работы.

In [ ]:
train_phrases = [
    "привет", "здравствуйте", "добрый день", "привет всем", "доброе утро", "хай", "здравствуй бот", "добрый вечер",
    "когда занятие", "во сколько начинается кружок", "какое расписание", "в какой день занятия",
    "когда следующий урок", "занятие в четверг", "во сколько урок", "расписание на неделю",
    "где проходит кружок", "в каком кабинете занятие", "как найти кабинет", "куда идти на урок",
    "где находится дворец", "какой адрес", "в каком корпусе кружок", "где аудитория",
    "что задали", "какое домашнее задание", "что делать дома", "есть домашка",
    "какая домашка на четверг", "задание на дом", "что нужно сделать к уроку", "где найти задание",
    "пока", "до свидания", "до встречи", "увидимся", "всего хорошего", "спасибо пока", "до завтра", "всем пока",
]
train_intents = (
    ["приветствие"] * 8 + ["расписание"] * 8 + ["место"] * 8 + ["домашка"] * 8 + ["прощание"] * 8
)
print("Фраз:", len(train_phrases), "| меток:", len(train_intents))

`["приветствие"] * 8` — список из восьми одинаковых элементов (умножение списка работает как умножение строки из урока 3), а `+` склеивает списки.

**Важно:** тестовые фразы из раздела 2 в обучение **не** входят — это наш test.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_phrases)
print("Таблица: фраз × слов в словаре =", X_train.shape)
print("Первые слова словаря:", vectorizer.get_feature_names_out()[:12])

In [ ]:
example = vectorizer.transform(["когда занятие в четверг"])
print(example.toarray())

`toarray()` показывает строку целиком: почти везде нули, а единицы — у слов «когда», «занятие», «четверг». Такая строка из 62 чисел и есть фраза «глазами модели». Слово «в» в словарь не попало: `CountVectorizer` по умолчанию пропускает однобуквенные слова.

## 5. Бот на машинном обучении

Классификатор возьмём знакомый — **kNN** (урок 20): для новой фразы он найдёт самые похожие фразы из обучения. Для текстов похожесть удобнее измерять не обычным расстоянием, а **косинусным** (`metric="cosine"`): оно смотрит, насколько совпадает **набор** слов, и не зависит от длины фразы.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3, metric="cosine")
knn.fit(X_train, train_intents)

def ml_intent(text):
    """Определить намерение с помощью обученной модели."""
    x = vectorizer.transform([text])
    return knn.predict(x)[0]

print("Доля верных ответов ML-бота:", round(run_tests(ml_intent), 2))

В `ml_intent` `transform` получает **список** фраз (у нас из одной), а `predict` возвращает массив ответов — по одному на фразу. `[0]` берёт первый и единственный ответ.

ML-бот ответил верно чаще, чем бот на правилах, но тоже ошибается. «Куда идти на занятие» он отнёс к расписанию: слово «занятие» в обучающих фразах чаще встречалось в вопросах о расписании. А «в какой день кружок» — к месту: слово «кружок» часто было в вопросах «где проходит кружок». **Модель знает слова, но не понимает смысла** — она ищет фразы с похожим набором слов.

### А если вопрос не по теме?

In [ ]:
for phrase in ["какая погода завтра", "сколько будет дважды два"]:
    print(f"{phrase:28} → {ml_intent(phrase)}")

Модель **всегда** выбирает один из пяти классов — даже на вопрос не по теме. Это опасно: бот уверенно отвечает невпопад. Нужен **порог уверенности** (урок 21): `predict_proba` у kNN показывает, какая доля соседей проголосовала за класс. Если уверенность низкая — честно отвечаем «не понял».

In [ ]:
def safe_intent(text, threshold=0.6):
    """Намерение или «не понял», если модель не уверена."""
    x = vectorizer.transform([text])
    if x.sum() == 0:                       # ни одного знакомого слова
        return "не понял"
    confidence = knn.predict_proba(x).max()
    if confidence < threshold:
        return "не понял"
    return knn.predict(x)[0]

for phrase in ["какая погода завтра", "в каком кабинете урок", "абракадабра"]:
    intent = safe_intent(phrase)
    print(f"{phrase:24} → {intent:12} → {replies[intent]}")

print("Доля верных на тестах:", round(run_tests(safe_intent), 2))

`predict_proba(x).max()` — самая большая из вероятностей, то есть доля соседей, проголосовавших за победивший класс. При `k = 3` это бывает 1 (все трое согласны), 0.67 (двое из трёх) или 0.33 (все разные). `x.sum() == 0` значит, что в строке-векторе одни нули: в фразе нет ни одного слова из словаря.

Порог — снова баланс: выше порог — меньше ответов невпопад, но и больше «не понял» на нормальные вопросы.

## 6. Правила, ML или большая языковая модель?

| | Правила | ML-классификатор | Большая языковая модель (LLM) |
|--|---------|------------------|-------------------------------|
| Что нужно | списки слов | размеченные примеры | готовая модель и хороший промпт |
| Понимает новые формулировки | плохо | средне | хорошо |
| Предсказуемость ответов | полная | высокая: ответы заготовлены | низкая: может выдумать |
| Объяснимость | полная | средняя | низкая |
| Риск выдумки | нет | нет | **есть** (галлюцинации) |
| Где уместно | 5–10 простых команд | FAQ, поддержка | свободный диалог под контролем |

Наш ML-бот только **выбирает** заготовленный ответ — поэтому он никогда не выдумает неверное расписание. Языковая модель **генерирует** ответ сама — гибко, но может уверенно ошибиться. Как устроены такие модели и как проверять их ответы — следующий урок.

### Безопасность чат-бота

- Бот должен честно говорить «не понял» и предлагать связаться с человеком.
- Бот не должен притворяться человеком.
- Бот не должен просить и хранить лишние личные данные.
- Важные ответы (оценки, медицина, деньги) — только от человека или из проверенного источника.

## 7. Практика

### Практика 1. Почини правила (7 минут)

Добавь в `rule_intent` слова так, чтобы бот на правилах проходил **все** тесты. Сколько слов пришлось добавить? Придумай ещё 3 новые тестовые фразы — проходят ли они?

In [ ]:
# Практика 1: скопируй rule_intent сюда, улучши и запусти run_tests

### Практика 2. Новое намерение (10 минут)

Научи ML-бота новому намерению **«учитель»** (например, «кто ведёт кружок», «как зовут преподавателя»):

1. Добавь в `train_phrases` 6–8 фраз и столько же меток в `train_intents`.
2. Добавь ответ в `replies`.
3. Переобучи `vectorizer` и `knn` и проверь бота на 3 новых фразах, которых нет в обучении.

In [ ]:
# Практика 2

<details>
<summary>Решение</summary>

```python
new_phrases = ["кто ведёт кружок", "как зовут преподавателя", "кто учитель", "имя учителя",
               "кто преподаёт", "кто руководитель кружка"]
train_phrases = train_phrases + new_phrases
train_intents = train_intents + ["учитель"] * len(new_phrases)
replies["учитель"] = "Кружок ведёт Анна Сергеевна."

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_phrases)
knn = KNeighborsClassifier(n_neighbors=3, metric="cosine")
knn.fit(X_train, train_intents)

for phrase in ["кто ведёт урок", "как зовут учителя", "а кто руководитель"]:
    intent = safe_intent(phrase)
    print(phrase, "→", intent, "→", replies[intent])
```
Обрати внимание на «как зовут учителя»: бот угадал благодаря словам «как зовут». А само слово «учителя» он не знает: в обучении было «учитель», и для мешка слов это **разные** слова. Это слабость простого подхода. В настоящих системах слова приводят к начальной форме или используют более умные модели.
</details>

### Практика 3. Поломай бота (5 минут)

Найди 3 фразы, на которых ML-бот ошибается или отвечает невпопад. Для каждой объясни **почему** (нет слов в словаре? похожие слова в другом намерении? опечатка?).

| Фраза | Ответ бота | Почему ошибся |
|-------|------------|---------------|
| | | |

## Работа с ИИ-помощником

- «Объясни, что такое мешок слов и почему он не учитывает порядок слов.»
- «Почему мой бот путает „учитель“ и „учителя“? Как это исправляют?»
- «Придумай 10 разных формулировок вопроса про расписание для тестирования бота» — это хороший способ получить **тестовые** фразы. Но проверяй их сам: не все придуманные фразы реалистичны.

## Мини-глоссарий урока 27

| Термин | Смысл |
|--------|-------|
| **Намерение (intent)** | чего хочет пользователь своей фразой |
| **Мешок слов** | текст → сколько раз встречается каждое слово, без порядка |
| **Словарь (vocabulary)** | все слова, которые знает модель |
| **`CountVectorizer`** | превращает тексты в таблицу «фраза × слово» |
| **Косинусное расстояние** | похожесть наборов слов, не зависит от длины текста |
| **Порог уверенности** | ниже порога бот отвечает «не понял» |

## Итог урока

1. Чат-бот = понять намерение (классификация текста) + выбрать ответ.
2. Мешок слов превращает текст в числа; дальше работают обычные модели.
3. Правила предсказуемы, но хрупки; ML понимает больше формулировок; порог «не понял» защищает от ответов невпопад.

## Домашнее задание (20 минут)

Придумай бота для своей темы (библиотека, спортивная секция, игра, школьная столовая): 3–4 намерения, по 6 обучающих фраз на каждое, ответы. Обучи ML-бота и проверь его на 5 новых фразах. Запиши, где он ошибается и почему. Это может стать основой итогового проекта.

In [ ]:
# Домашнее задание

## Самопроверка

1. Что такое намерение пользователя?
2. Почему бот на правилах ломается на новых формулировках?
3. Как фраза превращается в строку чисел в мешке слов?
4. Зачем боту ответ «не понял»?
5. Чем ML-бот с заготовленными ответами безопаснее бота на языковой модели?

**Следующий урок (01.04.2027):** большие языковые модели — как они устроены и как проверять их ответы.